In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='muted')

print('Libraries loaded.')

In [ ]:
df = pd.read_csv('../data/matches_prepared.csv')

print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')

---
## Section 1: Goal Scoring Evolution

How has goal scoring changed across 150 years? We use the `total_goals` feature engineered in Phase 2.

In [ ]:
# Average goals per match by decade
avg_goals_by_decade = df.groupby('decade')['total_goals'].mean().reset_index()
avg_goals_by_decade.columns = ['decade', 'avg_goals']
avg_goals_by_decade['avg_goals'] = avg_goals_by_decade['avg_goals'].round(2)

# Plot 
fig = px.line(
    avg_goals_by_decade,
    x='decade',
    y='avg_goals',
    title='Average goals per match by decade'
)
fig.show()

In [ ]:
# Total goals scored per decade
total_goals_by_decade = df.groupby('decade')['total_goals'].sum().reset_index()
total_goals_by_decade.columns = ['decade', 'total_goals']

# Plot 
fig = px.line(
    total_goals_by_decade,
    x='decade',
    y='total_goals',
    title='Total goals scored per decade'
)
fig.show()

In [ ]:
# Goal distribution — compare early era vs modern era using histograms
early_era  = df[df['year'] < 1950]['total_goals']
modern_era = df[df['year'] >= 1950]['total_goals']

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].hist(early_era, color='steelblue')
ax[0].set_title('Goal Distribution — Pre-1950',fontweight='bold')
ax[0].set_xlabel('Total Goals per Match')
ax[0].set_ylabel('Number of Matches')

ax[1].hist(modern_era,color='darkorange')
ax[1].set_title('Goal Distribution — 1950 Onward',fontweight='bold')
ax[1].set_xlabel('Total Goals per Match')
ax[1].set_ylabel('Number of Matches')

plt.suptitle('Goal Distribution: Early Era vs Modern Era',fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Pre-1950  avg goals per match : {early_era.mean():.2f}')
print(f'1950+     avg goals per match : {modern_era.mean():.2f}')

### Key Findings — Goal Scoring Evolution

- **Early international football was high-scoring.** Pre-1900 and early 1900s matches had average goals above 4.0 — partly because teams were unevenly matched and defensive tactics were not yet developed.
- **Scoring declined steadily from the 1930s to the 1980s**, as defensive organization, tactical sophistication, and professional training improved.
- **Modern football (post-1990) stabilizes around 2.5–3.0 goals per match** — reflecting well-organized defenses and more evenly matched international teams.
- Total goals per decade rises in absolute terms purely due to **more matches being played**, not higher per-match scoring.


---
## Section 2: Match Competitiveness Evolution

We use `goal_difference_abs` from Phase 2 to measure how one-sided matches have been across eras.

In [ ]:
# Average goal difference (absolute) by decade — measures typical winning margin
avg_gd_by_decade = df.groupby('decade')['goal_difference_abs'].mean().reset_index()
avg_gd_by_decade.columns = ['decade', 'avg_goal_diff']
avg_gd_by_decade['avg_goal_diff'] = avg_gd_by_decade['avg_goal_diff'].round(2)

# Plot 
fig = px.line(
    avg_gd_by_decade,
    x='decade',
    y='avg_goal_diff',
    title='Average GD by decade'
)
fig.show()

In [ ]:
# Classify matches into competitiveness categories using goal_difference_abs
# Draw = 0, Competitive = 1, Moderate Win = 2–3, Heavy Win = 4+
def classify_competitiveness(gd_abs):
    if gd_abs == 0:
        return 'Draw'
    elif gd_abs == 1:
        return 'Competitive Match'
    elif gd_abs <= 3:
        return 'Moderate Win'
    else:
        return 'Heavy Win'

df['competitiveness'] = df['goal_difference_abs'].apply(classify_competitiveness)
df['competitiveness'].value_counts()

In [ ]:
# Competitiveness category share by decade
comp_by_decade = df.groupby(['decade', 'competitiveness']).size().reset_index(name="matches")
print(comp_by_decade.columns)

# Plot 
fig = px.histogram(
    comp_by_decade,
    x='decade',
    y='matches',
    color='competitiveness',
    barmode='group',
    title='Competitiveness Categories by Decade'
)
fig.show()

### Key Findings — Match Competitiveness

- **Early eras (pre-1920) had far more heavy wins** — large goal margins were common when football was dominated by a handful of strong nations playing weaker opponents.
- **Average goal difference has declined steadily since 1900**, confirming that matches are becoming more competitive over time.
- **Modern football (post-1990) shows the lowest average goal difference** in the dataset's history — roughly 1.6–1.7 goals per match.
- **Competitive matches (1-goal margin) and draws now make up a larger share** of results than in any previous era.

---
## Section 3: Match Result Evolution

Using `match_result` from Phase 2, we track how the balance between home wins, away wins, and draws has shifted over time.

In [ ]:
# Match result counts by decade
result_by_decade = df.groupby(['decade', 'match_result']).size().reset_index(name='matches')

# Percentage share of match_result within each decade
result_by_decade['pct_match_result'] = result_by_decade.groupby('decade')['matches'].transform(lambda x: x / x.sum() * 100)
result_by_decade['pct_match_result'] = result_by_decade['pct_match_result'].round(2)

# Overall percentages across the whole dataset
overall_pct = df['match_result'].value_counts(normalize=True).mul(100).round(2).reset_index()
overall_pct.columns = ['match_result', 'pct_match_result']

# Plot 
fig = px.bar(
    result_by_decade,
    x='decade',
    y='matches',
    color='match_result',
    barmode='group',
    text = 'pct_match_result',
    title='Match result counts by decade',
    labels={'matches': 'Number of matches', 'decade': 'Decade'}
)
fig.show()




### Key Findings — Match Result Evolution

- **Home wins have consistently been the most common outcome** roughly 45-50% across every decade — home advantage is a persistent feature of international football.
- **Away wins mostly remained constant** roughly 25-30% across every decade. Away matches have always been a disadvantage 
- **Draws have remained relatively stable** — typically accounting for 20–25% of results across most eras.


---
## Section 4: High Scoring Match Trends

Using the `high_scoring_match` feature (Yes if total_goals >= 4, No otherwise) to track how the frequency of open, attacking games has changed.

In [ ]:
# Percentage of high-scoring matches by decade
hs_by_decade = df.groupby('decade')['high_scoring_match'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
hs_by_decade.columns = ['decade', 'pct_high_scoring']
hs_by_decade['pct_high_scoring'] = hs_by_decade['pct_high_scoring'].round(2)

# Plot 
fig = px.bar(
    hs_by_decade,
    x='decade',
    y='pct_high_scoring',
    text = 'pct_high_scoring',
    title='High-Scoring Matches (≥4 Goals)',
    labels={'pct_high_scoring': 'Percentage', 'decade': 'Decade'}
)
fig.show()



### Key Findings — High Scoring Match Trends

- **Early eras (pre-1950) had a dramatically higher share of high-scoring matches** — often exceeding 50% of games.
- The percentage has **declined significantly** since the 1980s, stabilizing around 25–30% in the modern era.
- This aligns with the findings from Section 1 — average goals per match have fallen as football became more competitive.
- Even in the modern era, roughly **1 in 4 matches** still produces 4 or more goals.

---
## Section 5: Tournament Evolution

How has the international competition landscape grown over time? We look at the number of unique tournaments per decade and which competitions dominate the dataset.

In [ ]:
# Unique tournaments per decade
tournaments_by_decade = df.groupby('decade')['tournament'].nunique().reset_index()
tournaments_by_decade.columns = ['decade', 'unique_tournaments']

fig = px.bar(
    tournaments_by_decade,
    x =  'decade',
    y = 'unique_tournaments',
    text = 'unique_tournaments',
    title='Unique tournaments per decade',
    labels={'unique_tournaments': 'Unique Tournaments', 'decade': 'Decade'}
)
fig.show()

In [ ]:
# Top tournaments — only include those with 20+ matches 
MIN_MATCHES = 20

tournament_counts = df['tournament'].value_counts().reset_index()
tournament_counts.columns = ['tournament', 'match_count']
tournament_counts = tournament_counts[tournament_counts['match_count'] >= MIN_MATCHES]

top20_tournaments = tournament_counts.head(20)

# Plot 
fig = px.bar(
    top20_tournaments,
    x =  'tournament',
    y = 'match_count',
    text = 'match_count',
    title='Unique tournaments per decade',
    labels={'tournament': 'Tournaments', 'match_count': 'Matches'}
)
fig.show()


In [ ]:
# Distribution of Friendly matches over decades
friendly_by_decade = (
    df[df['tournament'] == 'Friendly']
    .groupby('decade')
    .size()
    .reset_index(name='friendly_matches')
)

fig = px.bar(
    friendly_by_decade,
    x='decade',
    y='friendly_matches',
    text='friendly_matches',
    title='Friendly Matches per Decade',
    labels={'friendly_matches': 'Friendly matches','decade': 'Decade'}
)
fig.show()

### Key Findings — Tournament Evolution

- The **number of unique tournaments has grown dramatically**, especially from the 1970s onward, as FIFA and regional confederations introduced more structured competitions.
- **Friendly matches remain the most common tournament type** in the dataset — but their relative share has decreased by roughly half from 2010 to 2020 as official competitions expanded.


---
## Section 6: Neutral Venue Trends

Using `is_neutral` (1 = neutral ground, 0 = home/away ground) to track how often international matches are played at neutral venues.

In [ ]:
# Overall neutral vs non-neutral split
neutral_counts = df['is_neutral'].value_counts()
neutral_pct = df['is_neutral'].value_counts(normalize=True).mul(100).round(1)

print('Neutral vs Non-Neutral matches:')
pd.DataFrame({'Count': neutral_counts, 'Percentage': neutral_pct}).rename(
    index={0: 'Non-Neutral (Home/Away)', 1: 'Neutral Venue'})

In [ ]:
# Percentage of neutral matches by decade
neutral_by_decade = df.groupby('decade')['is_neutral'].mean().mul(100).round(1).reset_index()
neutral_by_decade.columns = ['decade', 'pct_neutral']

fig = px.bar(
    neutral_by_decade,
    x='decade',
    y='pct_neutral',
    text='pct_neutral',
    title='Percentage of neutral matches by decade',
    labels={'pct_neutral': 'Neutral Matches','decade': 'Decade'}
)
fig.show()

In [ ]:

neutral_by_decade['pct_non_neutral'] = 100 - neutral_by_decade['pct_neutral']

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(neutral_by_decade['decade'], neutral_by_decade['pct_non_neutral'],
       label='Home / Away Venue', color='steelblue')

ax.bar(neutral_by_decade['decade'], neutral_by_decade['pct_neutral'],
        label='Neutral Venue', color='coral')

ax.set_title('Neutral vs Non-Neutral Matches by Decade (%)', fontweight='bold')
ax.set_xlabel('Decade')
ax.set_ylabel('Percentage of Matches (%)')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Key Findings — Neutral Venue Trends

- **Neutral venue matches have always been a minority** — the majority of international football is played at one team's home ground.
- **There has been an increase in neutral venue matches** — roughlt 1 in 3 matches are played in neutral venue matches in deacde 2020.
- The **FIFA World Cup and continental championships** are the main drivers of neutral ground spikes — all matches are played at one host nation's stadiums.



---
## Section 7: Most Active Years and Decades

In [ ]:
# Most active years
print('Top 15 Most Active Years (by number of matches):')
most_active_years = df.groupby('year').size().reset_index(name='matches')
most_active_years.nlargest(15, 'matches').reset_index(drop=True)

**Commentary:** The most active years are almost all **post-2000**, reflecting the dense international calendar now in place — FIFA World Cup cycles generate qualification rounds, friendlies, and continental competitions stacked across every year.

In [ ]:
# Most active decades — total matches
print('Most Active Decades (by number of matches):')
df.groupby('decade').size().reset_index(name='matches').sort_values('matches', ascending=False).reset_index(drop=True)

**Commentary:** The **2000s and 2010s** are the most active decades overall — driven by FIFA's expansion to 32 teams in the 1998 World Cup, growing African and Asian football, and a packed international schedule. 